# Libraries

In [1]:
import numpy as np
import pandas as pd
import os
import h5py

In [2]:
import tensorflow as tf 
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import Model
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import train_test_split
# tf.config.run_functions_eagerly(False)
import random

I0000 00:00:1776748599.186903    1982 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# Set GPU

In [3]:
import tensorflow as tf #Buil model machine learning
import matplotlib.pyplot as plt
import pickle as pkl
import os
# tf.compat.v1.set_random_seed(1)

#Check GPU is using
gpus = tf.config.experimental.list_physical_devices('GPU')
print(gpus)
if gpus: #nếu có gpu
    try:
#         chọn 1 số cái visible
        tf.config.experimental.set_visible_devices(gpus[1], 'GPU') # gpus[index]
        
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
2 Physical GPUs, 1 Logical GPUs


I0000 00:00:1776751291.860393    1982 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9055 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:41:00.0, compute capability: 7.5


# Preprocessing data

In [4]:
data_path = 'D1'
save_path = 'Save_model/TestModel'
dir_list = ['CORONA', 'FLOATING', 'VOID', 'PARTICLE', 'NEW_NOISE']

In [5]:
test_ratio = 0.1
unlab_ratio = 0.80
ratio_change = 0.25

In [6]:
list_all_data = {}

for dir_name in dir_list:
    list_raw_array = []

    dir_name_full = os.path.join(data_path, dir_name)
    for file_name in os.listdir(dir_name_full):
        # if '.csv' in file_name:
        if file_name.endswith('.csv'):
            raw_df = pd.read_csv(os.path.join(dir_name_full, file_name), sep = ',', header = None)
            raw_array = raw_df.values
            raw_array_tensor = tf.convert_to_tensor(raw_array)
            raw_array_reshape = tf.reshape(raw_array_tensor, (3600, 128))
            list_raw_array.append(raw_array_reshape)
        
        list_all_data[dir_name] = list_raw_array

In [7]:
for dir_name in dir_list:
    print(dir_name, len(list_all_data[dir_name]))

CORONA 94
FLOATING 35
VOID 242
PARTICLE 66
NEW_NOISE 298


In [8]:
X_train = []
y_train = []
X_test = []
y_test = []
X_unlab = []
X_full = []
y_full = []
for i in range(len(dir_list)):
    key = dir_list[i]
    print(key)
    
    fault = list_all_data[key]
    random.shuffle(fault)
    test_amount = int(test_ratio * len(fault))
    
    test_dataset_amount = fault[:test_amount]
    train_dataset_amount = fault[test_amount:]
    # test_dataset = np.array(test_dataset_amount)
    # train_dataset = np.array(train_dataset_amount)
    test_dataset = np.concatenate([np.array(item) for item in test_dataset_amount], axis=0)
    train_dataset = np.concatenate([np.array(item) for item in train_dataset_amount], axis=0)
    

    unlab_amount = int(unlab_ratio * train_dataset.shape[0])
    unlab_train = np.array(train_dataset_amount[:unlab_amount])
    X_lab_train = np.array(train_dataset_amount[unlab_amount:])
    
    print(f"Test set ", test_dataset.shape)
    print(f"Train set", X_lab_train.shape)
    

    lab_train = np.full(X_lab_train.shape[0],i,dtype='uint8')
    lab_test = np.full(test_dataset.shape[0],i, dtype='uint8')
   
    y_onehot_train = np.eye(5)[lab_train]
    y_onehot_test = np.eye(5)[lab_test]



    X_train.append(X_lab_train)
    X_unlab.append(unlab_train)
    y_train.append(y_onehot_train)
    X_test.append(test_dataset)
    y_test.append(y_onehot_test)


combined_X_train = np.concatenate(X_train, axis =0)
print(f"------Total train", combined_X_train.shape)

combined_X_unlab = np.concatenate(X_unlab, axis =0)
print(f"------Total unlabel train", combined_X_unlab.shape)

combined_label_train= np.concatenate(y_train, axis =0)
print(f"------Total label train", combined_label_train.shape)

combined_X_test = np.concatenate(X_test, axis =0)
print(f"______Total test", combined_X_test.shape)

combined_label_test = np.concatenate(y_test,axis =0)
print(f"______Total label test", combined_label_test.shape)

CORONA
Test set  (32400, 128)
Train set (0,)
FLOATING
Test set  (10800, 128)
Train set (0,)
VOID
Test set  (86400, 128)
Train set (0,)
PARTICLE
Test set  (21600, 128)
Train set (0,)
NEW_NOISE
Test set  (104400, 128)
Train set (0,)
------Total train (0,)
------Total unlabel train (664, 3600, 128)
------Total label train (0, 5)
______Total test (255600, 128)
______Total label test (255600, 5)


In [9]:
class DefaultDataset(tf.data.Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y
        assert len(self.X) == len(self.Y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, index):
        _x = self.X[index]
        _y = self.Y[index]
        return _x, _y

# Data Augmentation

In [10]:
# Scale noise following Gaussian distribution
def scale_noise(X):
    mu, sigma = 1.0, 0.01
    noise = tf.random.normal(shape=tf.shape(X), mean=mu, stddev=sigma, dtype= tf.float32)
    X = tf.cast(X, dtype=tf.float32) * noise
    # noise = np.random.normal(mu,sigma,X.shape)
    # X = X * noise
    return X

In [11]:
import random
def random_crop(X):
    X = tf.convert_to_tensor(X)  # Convert X to a TensorFlow tensor
    non_zero_indices = tf.where(tf.not_equal(X, 0))

    # Calculate the number of non-zero elements in X
    num_non_zero = tf.shape(non_zero_indices)[0]

    # Calculate the number of elements to replace (10% of non-zero elements)
    num_to_replace = tf.cast(tf.round(tf.cast(num_non_zero, dtype=tf.float32) * 0.1), dtype=tf.int32)

    # Ensure num_to_replace is at least 1
    num_to_replace = tf.maximum(num_to_replace, 1)

    # Randomly select 'num_to_replace' indices to change to zero
    shuffled_indices = tf.random.shuffle(non_zero_indices)
    indices_to_replace = shuffled_indices[:num_to_replace]

    # Create the updates tensor with the same shape as indices_to_replace
    updates = tf.zeros([num_to_replace], dtype=X.dtype)

    # Set the selected non-zero elements to zero
    X = tf.tensor_scatter_nd_update(X, indices_to_replace, updates)

    return X

# Hyparameters for the algorithm

In [12]:
# EPOCHS = 200
EPOCHS = 20
TEMPERATURE = 0.1
SEED = 42 # the embeddings during training to improve the contrastive loss function.
AUTO = tf.data.AUTOTUNE
LEARNING_RATE = 0.0001
WEIGHT_DECAY = 0.0005

# Calculate the batch sizes for labeled and unlabeled samples

In [13]:
X_label_tensor = tf.convert_to_tensor(combined_X_train)
X_label_tensor.shape

TensorShape([0])

In [ ]:
X_unlab_aug = scale_noise(combined_X_unlab)
print(X_unlab_aug.shape)

X_ulb_scale_tensor = tf.convert_to_tensor(X_unlab_aug)
print(type(X_ulb_scale_tensor))

X_unlab_change = random_crop(combined_X_unlab)
print(X_unlab_aug.shape)

X_ulb_change_tensor = tf.convert_to_tensor(X_unlab_change)
print(type(X_ulb_change_tensor))

In [ ]:
x_unlabeled_train = tf.concat((X_unlab_change, X_ulb_change_tensor),0)
x_unlabeled_train.shape

In [ ]:
y_train_tensor = tf.convert_to_tensor(combined_label_train)
print(y_train_tensor.shape)
print(type(y_train_tensor))

In [ ]:
print(combined_X_test.shape)
X_test_tensor = tf.convert_to_tensor(combined_X_test)
print(type(X_test_tensor))

In [ ]:
print(combined_label_test.shape)
y_test_tensor = tf.convert_to_tensor(combined_label_test)
print(type(y_test_tensor))
y_test = tf.math.argmax(y_test_tensor,1)
y_test

In [ ]:
# Define the batch size
DATASET_UNLABELED_SIZE = x_unlabeled_train.shape[0]
DATASET_LABELED_SIZE = X_label_tensor.shape[0]
unlabeled_batch_size = 25
labeled_batch_size = 10
steps_per_epoch = DATASET_LABELED_SIZE//labeled_batch_size
total_steps = EPOCHS * steps_per_epoch
print(f"Batch size: {unlabeled_batch_size} (unlabeled) + {labeled_batch_size} (labeled)")

# Load the unlabeled and labeled samples for training
unlabeled_train= tf.data.Dataset.from_tensor_slices(x_unlabeled_train)
unlabeled_train = (
    unlabeled_train.shuffle(buffer_size=5 * unlabeled_batch_size)
    .batch(unlabeled_batch_size)
    
)

# Convert labels to one-hot vectors              
labeled_train = tf.data.Dataset.from_tensor_slices((X_label_tensor,y_train_tensor))
labeled_train = (
    labeled_train.shuffle(buffer_size=5 * labeled_batch_size)
    .batch(labeled_batch_size)   
)

# Load the test samples
test = (
    tf.data.Dataset.from_tensor_slices((X_test_tensor,y_test_tensor))
    .batch(unlabeled_batch_size)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

# Combine the labeled and unlabeled samples for training
data = tf.data.Dataset.zip((unlabeled_train, labeled_train)).prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
# Define the batch size
DATASET_UNLABELED_SIZE = x_unlabeled_train.shape[0]
DATASET_LABELED_SIZE = X_label_tensor.shape[0]
unlabeled_batch_size = 25
labeled_batch_size = 10
steps_per_epoch = DATASET_LABELED_SIZE//labeled_batch_size
total_steps = EPOCHS * steps_per_epoch
print(f"Batch size: {unlabeled_batch_size} (unlabeled) + {labeled_batch_size} (labeled)")

# Load the unlabeled and labeled samples for training
unlabeled_train= tf.data.Dataset.from_tensor_slices(x_unlabeled_train)
unlabeled_train = (
    unlabeled_train.shuffle(buffer_size=5 * unlabeled_batch_size)
    .batch(unlabeled_batch_size)
    
)

# Convert labels to one-hot vectors              
labeled_train = tf.data.Dataset.from_tensor_slices((X_label_tensor,y_train_tensor))
labeled_train = (
    labeled_train.shuffle(buffer_size=5 * labeled_batch_size)
    .batch(labeled_batch_size)   
)

# Load the test samples
test = (
    tf.data.Dataset.from_tensor_slices((X_test_tensor,y_test))
    .batch(unlabeled_batch_size)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

# Combine the labeled and unlabeled samples for training
data = tf.data.Dataset.zip((unlabeled_train, labeled_train)).prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
data

# Define the CNN architecture and Baseline model


In [ ]:
def create_custom_model(input_shape, num_classes):
    # Define the input layer with the specified shape
    inputs = Input(shape=input_shape, name='input')
    
    # Convolutional layers
    x = Conv2D(16, (3, 3), activation='relu', padding='same', name='conv2d_1')(inputs)
    x = MaxPooling2D((2, 2), name='maxpooling_1')(x)
    # x = Dropout(0.3)(x)
    
    x = Conv2D(8, (3, 3), activation='relu', padding='same', name='conv2d_2')(x)
    x = MaxPooling2D((2, 2), name='maxpooling_2')(x)
    
    # Flatten the output
    x = Flatten(name='Flatten')(x)

    # # # Dense layers
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu', name='Dense_0')(x)

    x = Dropout(0.2)(x)
    
    # Output layer with softmax activation for multi-class classification
    outputs = Dense(num_classes,kernel_regularizer=regularizers.l2(WEIGHT_DECAY),name='output')(x)

    # Create the model by specifying inputs and outputs
    model = Model(inputs=inputs, outputs=outputs, name='Custom_model')
    
    return model

In [ ]:
input_shape= (3600, 128, 1)
num_classes = 5  
custom_model = create_custom_model(input_shape, num_classes)
custom_model.summary()

W0000 00:00:1776145855.859790   25750 nvptx_libdevice_path.cc:41] Can't find libdevice directory ${CUDA_DIR}/nvvm/libdevice. This may result in compilation or runtime failures, if the program we try to run uses routines from libdevice.
Searched for CUDA in the following directories:
  ./cuda_sdk_lib
  ipykernel_launcher.runfiles/cuda_nvcc
  ipykernel_launcher.runfiles/cuda_nvdisasm
  ipykernel_launcher.runfiles/nvidia_nvshmem
  ipykernel_launcher.runfiles/cuda_nvvm
  ipykernel_launcher.runfiles/cuda_cudart
  /usr/local/cuda
  /opt/cuda
  /mnt/c/Users/PC/Documents/Dan/aboutpaperreviews/.venv/lib/python3.12/site-packages/tensorflow/python/platform/../../../nvidia/cuda_nvcc
  /mnt/c/Users/PC/Documents/Dan/aboutpaperreviews/.venv/lib/python3.12/site-packages/tensorflow/python/platform/../../../../nvidia/cuda_nvcc
  /mnt/c/Users/PC/Documents/Dan/aboutpaperreviews/.venv/lib/python3.12/site-packages/tensorflow/python/platform/../../cuda
  /mnt/c/Users/PC/Documents/Dan/aboutpaperreviews/.venv/

NameError: name 'WEIGHT_DECAY' is not defined

# Loss computation utilities

In [ ]:
def compute_supervised_loss(y_labeled, y_predicted):
    loss_func = keras.losses.CategoricalCrossentropy(from_logits=True)
    supervised_loss = loss_func(y_labeled, y_predicted)
    return supervised_loss


def compute_unsupervised_loss(pseudo_labels, unlabeled_data_s, mask):
    loss_func = keras.losses.CategoricalCrossentropy(from_logits=True, reduction="none")
    # Do not update weight during training
    pseudo_labels = tf.stop_gradient(pseudo_labels)
    unsupervised_loss = loss_func(pseudo_labels, unlabeled_data_s)
    mask = tf.cast(mask, unsupervised_loss.dtype)
    unsupervised_loss *= mask
    return tf.reduce_mean(unsupervised_loss,0)

# Fixmatch model

In [ ]:
class FixMatch(keras.Model):
    def __init__(self,model,total_steps,tau = 0.9):
        super().__init__()
        self.model = model
        # Denotes the confidence threshold
        self.tau = tau             
        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.total_steps = total_steps
        self.current_step = tf.Variable(0,dtype="int64")
        
    @property
    def metrics(self):
        return [self.loss_tracker]

    def compute_mu(self):
        pi = tf.constant(np.pi, dtype="float32")
        step = tf.cast(self.current_step, dtype="float32")
        return 0.5 - tf.cos(tf.math.minimum(pi,(2*pi*step)/self.total_steps)) / 2

    def call(self, inputs, training=False):
        # Forward pass through our custom model
        return self.model(inputs, training=training)
        
    def train_step(self,data):
        unlabeled, (X_labeled, y_labeled) = data
        unlabeled = tf.cast(unlabeled, dtype=tf.float32)
        X_labeled = tf.cast(X_labeled, dtype=tf.float32)

        # Apply two different augmentation methods for input data
        labeled_dataset_w = scale_noise(X_labeled)
        unlabeled_dataset_w = random_crop(unlabeled)
        unlabeled_dataset_s = scale_noise(unlabeled)
          
        combined_data = tf.concat([labeled_dataset_w, unlabeled_dataset_w, unlabeled_dataset_s],0)
        total_labeled_data = tf.shape(labeled_dataset_w)[0]
        total_unlabeled_data = tf.shape(tf.concat([unlabeled_dataset_w, unlabeled_dataset_s], 0))[0]
   
        with tf.GradientTape() as tape:
             # Forward passes
            combined_logits = self.model(combined_data, training=True)
            z_d_prime_labeled_data = self.model(labeled_dataset_w, training=False)
            z_prime_labeled_data = combined_logits[:total_labeled_data]

            # Random logit interpolation for the labeled data
            lambd = tf.random.uniform((total_labeled_data, 5), 0, 1)
            final_labeled_data_logits = (lambd * z_prime_labeled_data) + ((1 - lambd) * z_d_prime_labeled_data)

            # Compute softmax for logits of the weakly augmented labeled data
            y_hat_labeled_data_w = tf.nn.softmax(final_labeled_data_logits[:tf.shape(labeled_dataset_w)[0]])

            # Extract logits for the weakly augmented unlabeled data
            logits_unlabeled = combined_logits[total_labeled_data:]
            logits_unlabeled_w = logits_unlabeled[:tf.shape(unlabeled_dataset_w)[0]]

            y_hat_unlabeled_data_w = tf.nn.softmax(logits_unlabeled_w)

            # Distribution alignment (only consider weakly augmented data)
            # Align the predicted label distribution to that of label data
            expectation_ratio = tf.reduce_mean(y_hat_unlabeled_data_w) / tf.reduce_mean(y_hat_labeled_data_w)
            y_tilde_unlabeled_data_w = tf.math.l2_normalize(y_hat_unlabeled_data_w * expectation_ratio, 1)
       
            # Relative confidence thresholding
            row_wise_max = tf.reduce_max(y_hat_labeled_data_w, axis=-1)
            final_sum = tf.reduce_mean(row_wise_max, 0)
            c_tau = self.tau * final_sum
            # c_tau = self.tau
            mask = tf.reduce_max(y_tilde_unlabeled_data_w, axis=-1) >= c_tau

            # Supervised loss
            supervised_loss = compute_supervised_loss(y_labeled, final_labeled_data_logits[:tf.shape(labeled_dataset_w)[0]])
            
            # Unsupervised loss
            unsupervised_loss = compute_unsupervised_loss(
                y_hat_unlabeled_data_w,
                logits_unlabeled[tf.shape(y_hat_unlabeled_data_w)[0]:],
                mask
            )
            
            t = self.compute_mu()
            total_loss = supervised_loss + (t * unsupervised_loss)
            self.current_step.assign_add(1) 
            
        gradients = tape.gradient(total_loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.model.trainable_variables))

        self.loss_tracker.update_state(total_loss)
        return {"loss":self.loss_tracker.result()}


# Instantiate Fixmatch model and compile it

In [ ]:
reduce_lr = keras.optimizers.schedules.CosineDecay(LEARNING_RATE, total_steps , 0.25)
optimizer = keras.optimizers.SGD(reduce_lr, momentum=0.9)

fixmatch_trainer = FixMatch(model=custom_model, total_steps=total_steps)
fixmatch_trainer.compile(optimizer=optimizer)

# Model training

In [ ]:
import os 
checkpoint_path = "training_7/my_checkpoints"
checkpoint_dir = os.path.dirname(checkpoint_path)

In [ ]:
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,
    monitor='loss',
    mode='min',
    save_best_only=True,
    verbose = 0)

In [ ]:
fixmatch_trainer.fit(data, epochs=EPOCHS, callbacks=[model_checkpoint_callback])

Epoch 1/200
11/11 [==============================] - 2s 131ms/step - loss: 3.7239
Epoch 2/200
11/11 [==============================] - 1s 129ms/step - loss: 1.6281
Epoch 3/200
11/11 [==============================] - 2s 142ms/step - loss: 0.6743
Epoch 4/200
11/11 [==============================] - 1s 127ms/step - loss: 0.4166
Epoch 5/200
 2/11 [====>.........................] - ETA: 1s - loss: 0.2079

In [ ]:
os.listdir(checkpoint_dir)

In [ ]:
# fixmatch_trainer.load_weights(checkpoint_path)

# Evaluation on the test set

In [ ]:
fixmatch_trained_model = fixmatch_trainer.model
fixmatch_trained_model.compile(metrics=keras.metrics.SparseCategoricalAccuracy())

_, accuracy = fixmatch_trained_model.evaluate(test)
print(f"Accuracy on  test set: {accuracy * 100:.2f}%")

In [ ]:
y_predict = fixmatch_trained_model.predict(X_test_tensor)

In [ ]:
y_predict = np.argmax(y_predict, axis = 1)

In [ ]:
y_test

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
label_names = ['Corona', 'Floating', 'Void', 'Particle', 'Noise']
print(classification_report(y_test, y_predict, target_names =label_names ))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
figure = confusion_matrix(y_test, y_predict)

# Define class labels
label_names = ['Corona', 'Floating', 'Void', 'Particle', 'Noise']

fig, ax = plt.subplots(figsize=(4, 4))
display = ConfusionMatrixDisplay(figure, display_labels=label_names)
display.plot(cmap=plt.cm.Blues, ax=ax, values_format='d') 

# Rotate x-axis labels for better readability
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Add labels to the axis
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")

# Show the plot
plt.tight_layout()  # Adjust layout for better spacing
# Save the figure after it has been shown
plt.savefig('Confusion_matrix_New2.png', transparent =True,bbox_inches='tight', dpi=350, format='png')
plt.show()